# সহজ ভাষায় Notebook Guide

এই notebook-এ theory এবং code পাশাপাশি শেখানো হয়েছে। Technical term English-এ থাকবে, আর explanation Bangla-তে—যাতে code-এর language-এর সাথে পরিচিত থেকেও concept সহজে বোঝা যায়।

## কীভাবে ব্যবহার করবেন?

1. Cell উপর থেকে নিচে sequence অনুযায়ী run করুন।
2. Run করার আগে expected output কী হতে পারে লিখে ভাবুন।
3. Output-এর metric, shape এবং visualization explanation-এর সাথে compare করুন।
4. Error হলে import, file path, data shape এবং dependency একে একে check করুন।
5. Notebook শেষে নিজের ভাষায় লিখুন: এটি কোন problem solve করেছে, কীভাবে করেছে এবং limitation কী।

> **Important:** Notebook-এর সব cell successful run হলেই result correct প্রমাণ হয় না; data leakage, wrong assumption এবং misleading metric-ও validate করতে হবে।

# 01 - MNIST EDA & Optimization Comparison Preview

This notebook does two things:

1. **Exploratory data analysis (executed)** — load the raw MNIST dataset via `src/data/loader.py`, inspect shapes, class balance, and a sample grid of digits.
2. **Three-way comparison preview (markdown only, not executed here)** — an explanation of what the Baseline vs. Dropout vs. Fully Optimized training-curve comparison in `app.py`'s **Train & Compare** tab will show, and why. Actual training happens in the Streamlit app or `src/training/trainer.py`, not in this notebook, since it can take several minutes on the full dataset.

TEACHING NOTE: keeping EDA and training in separate places is deliberate. EDA should be fast, cheap, and safe to re-run anytime you touch the data pipeline. Training is expensive and stateful (it produces model artifacts). Mixing them into one notebook makes it tempting to accidentally re-run a 10-minute training cell just to re-look at a data histogram.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable so `import config` and `import src...`
# work the same way here as they do from app.py / tests, even though this
# notebook lives one directory down in notebooks/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np

from config import get_config
from src.data.loader import load_mnist

CONFIG = get_config()
data = load_mnist()

print("train:", data.x_train.shape, data.y_train.shape)
print("val:  ", data.x_val.shape, data.y_val.shape)
print("test: ", data.x_test.shape, data.y_test.shape)
print("pixel range:", data.x_train.min(), "to", data.x_train.max())

## Class balance

MNIST is close to balanced across all 10 digit classes, which matters for this project: it means plain `accuracy` is a trustworthy metric here (no class is rare enough that a model could game accuracy by ignoring it), so we don't need class weighting or a precision/recall breakdown to interpret the results honestly.

In [ ]:
values, counts = np.unique(data.y_train, return_counts=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(values, counts, color="#4C72B0")
ax.set_xticks(values)
ax.set_xlabel("Digit")
ax.set_ylabel("Count (training split)")
ax.set_title("MNIST Training Set Class Balance")
plt.show()

## Sample grid

A quick visual sanity check that the loader's normalize/reshape pipeline (`src/data/loader.py::_normalize_and_reshape`) hasn't corrupted the images — pixel values in `[0, 1]`, correct orientation, recognizable digits.

In [ ]:
rng = np.random.default_rng(CONFIG.data.random_seed)
sample_idx = rng.choice(len(data.x_train), size=16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for ax, idx in zip(axes.flat, sample_idx):
    ax.imshow(data.x_train[idx].squeeze(), cmap="gray")
    ax.set_title(str(data.y_train[idx]), fontsize=10)
    ax.axis("off")
fig.suptitle("Random MNIST Training Samples")
plt.tight_layout()
plt.show()

## What the three-way comparison will show (not executed here)

The **Train & Compare** tab of `app.py` (backed by `src/training/trainer.py::compare_variants`) trains three CNNs that share an identical convolutional backbone and differ only in their regularization / training-loop strategy:

| Variant | Regularization | Training loop | Expected train/val accuracy gap |
|---|---|---|---|
| Baseline | None | Fixed epoch count | Largest — keeps growing the longer it trains |
| With Dropout | `Dropout(0.5)` before output | Fixed epoch count | Smaller, but still trains a fixed number of epochs regardless of overfitting signs |
| Fully Optimized | `BatchNorm` + `Dropout(0.3)` throughout | `EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)` + `ReduceLROnPlateau` | Smallest and most stable; training stops automatically once validation loss stops improving |

Expect to see, once you run the comparison in the app:

- **Baseline**: training accuracy climbing toward ~99.5-100% while validation accuracy plateaus noticeably lower — the textbook overfitting signature described in the Class 2 lecture (`../Class 2 Lecture/context.md`).
- **With Dropout**: a visibly narrower gap between the solid (train) and dashed (val) curves, because Dropout prevents individual neurons from memorizing specific training examples.
- **Fully Optimized**: the narrowest, most stable gap, training stopping automatically at whatever epoch validation loss stopped improving, and (on the full 60k-image training set) test accuracy exceeding 98%.

This notebook does not run that training itself — training all three variants on the full dataset takes several minutes even on a fast machine, and this notebook is meant to stay fast to re-run for EDA purposes. Use `app.py`'s **Train & Compare** tab (with the quick-demo subset for a fast pass, or the full dataset for lecture-accurate numbers) or call `src.training.trainer.compare_variants` directly from a script to reproduce the actual curves.